In [2]:
# This notebook aims to provide reproduction for most of the figures/tables. For TABLE III (main PA3 result) and TABLE IV (ablation study), there are separate scripts since they involve deep learning model training and result parsing.

# This cell provides Jupyter Notebook autoreload and dataset path configuration util functions.
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from pa3.utils.env import load_dotenv

load_dotenv()
PA3_REPO_ROOT = Path(os.environ["PA3_REPO_ROOT"]).expanduser()


def pa3_path(relative_path: str) -> str:
    return str(PA3_REPO_ROOT / relative_path)

In [ ]:
# Reproduction of Fig. 3 The BIH flow size patterns
import matplotlib.patches as mpatches

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root_dir = "../data_extract"


dfs = dict()
dfs['content-signature-2.cdn.mozilla.net'] = pd.read_csv(f"{root_dir}/content-signature-2.cdn.mozilla.net.csv")
dfs['firefox.settings.services.mozilla.com'] = pd.read_csv(f"{root_dir}/firefox.settings.services.mozilla.com.csv")
dfs['firefox-settings-attachments.cdn.mozilla.net'] = pd.read_csv(f"{root_dir}/firefox-settings-attachments.cdn.mozilla.net.csv")

# Each csv contains the same columns: trojan, vmess, shadowsocks
# For each csv, for each column, compute the histogram of the column, where the bins are of size 500.
arrays = {'content-signature-2.cdn.mozilla.net': [], 'firefox.settings.services.mozilla.com': [], 'firefox-settings-attachments.cdn.mozilla.net': []}
for sni in dfs:
    columns = []
    for proto in ['vmess', 'shadowsocks', 'trojan']:
        col = dfs[sni][proto].dropna().to_numpy().astype(np.int64)
        n = max(1, int(len(col) * .5))
        if n < len(col):
            col = np.random.choice(col, n, replace=False)
        columns.append(col)
    arrays[sni] = columns

protocols = ["VMess", "Shadowsocks", "Trojan"]
color_map = ["#B279A2", "#54A24B", "#F58518"]

patches = [mpatches.Patch(color=color_map[i], label=protocols[i]) for i in range(3)]

x_lim = [300000, 31_000, 4e6]
x_ticks = [
    [0, 100_000, 200_000, 300_000],     # 0, 100K, 200K, 300K
    [0, 10_000, 20_000, 30_000],     # 0, 10K, 20K, 30K
    [0, 1_000_000, 2_000_000, 3_000_000, 4_000_000]   # 0, 1M, 2M, 3M, 4M
]
x_ticklabels = [
    ['0', '100K', '200K', '300K'],
    ['0', '10K', '20K', '30K'],
    ['0', '1M', '2M', '3M', '4M']
]

fig, axes = plt.subplots(3, 3, figsize=(8, 4))

for j in range(3):
    for i, sni in enumerate(dfs):
        axes[i, j].hist(arrays[sni][j][arrays[sni][j] < x_lim[i]], bins=60, color=color_map[j], label=protocols[j])
        axes[i, j].set_xlim(0, x_lim[i])
        axes[i, j].tick_params(axis='x', labelsize=14)
        axes[i, j].set_yticks([])
        axes[i, j].set_xticks(x_ticks[i])
        axes[i, j].set_xticklabels(x_ticklabels[i])
        axes[i, 1].set_title(sni, fontsize=16)
            

# Manually create a legend for the whole figure
fig.legend(handles=patches, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.06), fontsize=14)

fig.subplots_adjust(hspace=0.9, wspace=0.2)
# plt.savefig('histograms.png')

plt.savefig('browser_noise.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Reproduction of Fig. 4 BIH v.s. website flow size distributions.
# Since this figure involves some complicated layout configuration, which might overrides some default Jupyter Notebook settings. It requires a "Kernel Restart" after running this cell for later reproduction. 

# Jupyter inline defaults to bbox_inches="tight" when displaying figures.
# That breaks inset_axes + mark_inset (huge raster). Use inline_bbox_none() only
# around plt.show() for the KDE/inset figure; other cells can use tight_layout normally.
from contextlib import contextmanager

import matplotlib as mpl


@contextmanager
def inline_bbox_none():
    """Temporarily disable tight bbox for Jupyter inline display (inset figures)."""
    from IPython import get_ipython

    ip = get_ipython()
    prev_inline = None
    prev_rc = mpl.rcParams["savefig.bbox"]
    if ip is not None:
        prev_inline = ip.config.get("InlineBackend.print_figure_kwargs")
        ip.run_line_magic(
            "config",
            "InlineBackend.print_figure_kwargs = {'bbox_inches': None}",
        )
    mpl.rcParams["savefig.bbox"] = "standard"
    try:
        yield
    finally:
        mpl.rcParams["savefig.bbox"] = prev_rc
        if ip is not None:
            if prev_inline is None:
                ip.run_line_magic(
                    "config",
                    "InlineBackend.print_figure_kwargs = {}",
                )
            else:
                ip.run_line_magic(
                    "config",
                    f"InlineBackend.print_figure_kwargs = {prev_inline!r}",
                )

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from scipy.stats import gaussian_kde

STAT_ROOT = "../data_extract"
PROTOCOLS = ["VMess", "Shadowsocks", "Trojan"]
FLOW_SIZE_ROOT = pa3_path("VisualSeg/flow_size")

# Canonical main-panel window up to 10⁷ (10M), matching requested tick labels.
# Tick at 10² is labelled "0" (true zero is invalid on a log-scale x-axis).
MAIN_X_TICKS = np.array([1e2, 1e3, 1e4, 1e5, 1e6, 1e7])
MAIN_X_TICK_LABELS = ["0", "1K", "10K", "100K", "1M", "10M"]
N_KDE_SAMPLES_LOG = 600
N_KDE_SAMPLES_LINEAR = 800
INSET_KDE_MAX = 100_000  # samples for inset KDE
INSET_X_DISPLAY_MAX = 30_000  # visible x-range in inset

BIH_HOSTNAMES = [
    "firefox-settings-attachments.cdn.mozilla.net",
    "firefox.settings.services.mozilla.com",
    "content-signature-2.cdn.mozilla.net",
]


def _load_bih_arrays(protocol: str):
    parts = []
    for host in BIH_HOSTNAMES:
        df = pd.read_csv(f"{STAT_ROOT}/{host}.csv")
        col = df[protocol].dropna().to_numpy(dtype=np.float64)
        parts.append(col)
    bih_concat = np.concatenate(parts) if parts else np.array([])
    services_only = pd.read_csv(
        f"{STAT_ROOT}/firefox.settings.services.mozilla.com.csv"
    )
    srv = services_only[protocol].dropna().to_numpy(dtype=np.float64)
    return bih_concat, srv


def safe_kde_plot(ax, samples, x_eval, markevery=None, **plot_kw):
    s = np.asarray(samples, dtype=float)
    s = s[np.isfinite(s)]
    if s.size == 0:
        return None
    if np.unique(np.round(s, 8)).size < 2:
        return None
    try:
        kde = gaussian_kde(s)
        if markevery is not None:
            plot_kw.setdefault("markevery", markevery)
        (line,) = ax.plot(x_eval, kde(x_eval), **plot_kw)
        return line
    except (np.linalg.LinAlgError, ValueError):
        return None


# layout=None: avoid auto layout heuristics conflicting with inset_axes children.
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), layout=None)
legend_handles = []

for idx, protocol in enumerate(PROTOCOLS):
    website_flow = np.load(f"{FLOW_SIZE_ROOT}/{protocol.lower()}.npz")["flow_size"].astype(np.float64)
    website_flow = website_flow[np.isfinite(website_flow)]

    bih_all, services_flow = _load_bih_arrays(protocol.lower())
    website_main = website_flow[website_flow > 0]
    bih_main = bih_all[bih_all > 0]

    # Always evaluate KDE on the full main-panel x window so curves are not
    # clipped when a protocol's samples start above 1K (e.g. trojan website min ~5K).
    x_vals_log = np.logspace(
        np.log10(MAIN_X_TICKS[0]),
        np.log10(MAIN_X_TICKS[-1]),
        N_KDE_SAMPLES_LOG,
    )

    ax = axes[idx]
    lw_main = safe_kde_plot(
        ax, 
        website_main, 
        x_vals_log, 
        color="#4C78A8", 
        linewidth=2, 
        linestyle="--",
        label="Website Flow", 
    )
    lb_main = safe_kde_plot(
        ax,
        bih_main,
        x_vals_log,
        color="#F58518",
        linewidth=2,
        label="BIH Flow",
    )
    if idx == 0:
        legend_handles.extend([lw_main, lb_main])

    ax.set_xscale("log")
    ax.set_xlim(MAIN_X_TICKS[0], MAIN_X_TICKS[-1])
    ax.set_xticks(MAIN_X_TICKS)
    ax.set_xticklabels(MAIN_X_TICK_LABELS, fontsize=16)
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())

    if idx == 0:
        ax.set_ylabel("Density", fontsize=16)
    ax.set_yticks([])
    # ax.grid(True, which="major", linestyle="--", alpha=0.35)
    # ax.grid(True, which="minor", linestyle=":", alpha=0.2)
    ax.set_title(protocol, fontsize=16)

    # Inset: KDE on flows <= INSET_KDE_MAX; display only 0–INSET_X_DISPLAY_MAX
    website_zoom = website_flow[website_flow <= INSET_KDE_MAX]
    services_zoom = services_flow[services_flow <= INSET_KDE_MAX]

    axins = inset_axes(
        ax,
        width="55%",
        height="45%",
        loc="lower left",
        bbox_to_anchor=(0.01, 0.11, 1.0, 1.0),
        bbox_transform=ax.transAxes,
        borderpad=1,
    )
    x_lin = np.linspace(0.0, INSET_KDE_MAX, N_KDE_SAMPLES_LINEAR)
    inset_markevery = max(1, len(x_lin) // 24)
    lw_zoom = safe_kde_plot(
        axins,
        website_zoom,
        x_lin,
        color="#4C78A8",
        linestyle="--",
        linewidth=1.5,
        marker="s",
        markersize=4,
        markevery=inset_markevery,
        label=r"Website Flow ($\leq$30K)",
    )
    lb_zoom = safe_kde_plot(
        axins,
        services_zoom,
        x_lin,
        color="#F58518",
        linewidth=1.5,
        marker="D",
        markersize=4,
        markevery=inset_markevery,
        label=r"BIH Flow ($\leq$30K)",
    )
    if idx == 0:
        legend_handles.extend([lw_zoom, lb_zoom])

    axins.set_xlim(0, INSET_X_DISPLAY_MAX)
    axins.set_xticks(np.arange(0, INSET_X_DISPLAY_MAX + 1, 10_000))
    axins.set_xticklabels(["0", "10K", "20K", "30K"], fontsize=16)
    axins.set_yticks([])
    axins.grid(True, axis="x", linestyle="--", alpha=0.35)

    # mark_inset breaks bbox_inches="tight" (Jupyter inline default); run cell 0 first.
    mark_inset(ax, axins, loc1=2, loc2=4, fc="none")

handles = [h for h in legend_handles if h is not None]
labels = [h.get_label() for h in handles]
if handles:
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=len(handles),
        fontsize=16,
        framealpha=0.9,
        columnspacing=.6
    )

fig.supxlabel("Flow Size", fontsize=16, y=0.00)
# Avoid tight_layout: inset_axes + Jupyter's tight-bbox raster can explode canvas size.
fig.subplots_adjust(left=0.06, right=0.94, top=0.78, bottom=0.18, wspace=0.16)

fig.savefig("browser_noise_2.pdf", bbox_inches=None)
with inline_bbox_none():
    plt.show()

In [ ]:
# Reproduction of Fig. 5 Server/Client Hello intra-flow indices
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches
from scipy import stats

fig, ax = plt.subplots(1, 2, figsize=(8, 2))
fig.subplots_adjust(wspace=0.25)  # Adjust horizontal distance between subplots

ch_df = pd.read_csv(pa3_path("VisualSeg/ch_search/database.csv"))
sh_df = pd.read_csv(pa3_path("VisualSeg/sh_search/database.csv"))

protocols = ['NoProxy', 'VMess', 'Shadowsocks', 'Trojan']
color_map = {"NoProxy": "#4C78A8", "VMess": "#B279A2", "Shadowsocks": "#54A24B", "Trojan": "#F58518"}

xticks = [0, 5, 10, 15, 20, 25]
xtick_labels = [str(x) for x in xticks]

# --- Client Hello (CH) subplot ---
y_max_ch = 0  # to self-decide y-lim for CH
for protocol in protocols:
    data = ch_df.query(f"{protocol.lower()} >= 0")[protocol.lower()].tolist()
    hist, edges = np.histogram(data, bins=20, range=(0, 20), density=True)
    kde = stats.gaussian_kde(data)
    x_vals = np.linspace(min(edges), max(edges), 500)
    y_vals = kde(x_vals)
    ax[0].plot(x_vals, y_vals, color=color_map[protocol], label=protocol, linewidth=1)
    ax[0].fill_between(x_vals, y_vals, alpha=0.3, color=color_map[protocol])
    y_max_ch = max(y_max_ch, np.max(y_vals))
ax[0].set_xlabel('Packet Index', fontsize=16)
ax[0].set_ylabel('Density', fontsize=16)
ax[0].set_title('Client Hello Index', fontsize=16)
ax[0].tick_params(axis='x', labelsize=14)
ax[0].set_xticks(xticks)
ax[0].set_xticklabels(xtick_labels)
ax[0].tick_params(axis='y', labelsize=14)
# Set 5 y-ticks and y-lim automatically based on data
ax[0].set_ylim(0, np.ceil(y_max_ch * 1.1 * 10) / 10 if y_max_ch > 0 else 1)
ax[0].yaxis.set_major_locator(plt.MaxNLocator(nbins=5))

# --- Server Hello (SH) subplot ---
y_max_sh = 0  # self-decide y-lim for SH
for protocol in protocols:
    data = sh_df.query(f"{protocol.lower()} >= 0")[protocol.lower()].tolist()
    hist, edges = np.histogram(data, bins=25, range=(0, 25), density=True)
    kde = stats.gaussian_kde(data)
    x_vals = np.linspace(min(edges), max(edges), 500)
    y_vals = kde(x_vals)
    ax[1].plot(x_vals, y_vals, color=color_map[protocol], label=protocol, linewidth=1)
    ax[1].fill_between(x_vals, y_vals, alpha=0.3, color=color_map[protocol])
    y_max_sh = max(y_max_sh, np.max(y_vals))
ax[1].set_xlabel('Packet Index', fontsize=16)
ax[1].set_ylabel('Density', fontsize=16)
ax[1].set_title('Server Hello Index', fontsize=16)
ax[1].tick_params(axis='x', labelsize=14)
ax[1].set_xticks(xticks)
ax[1].set_xticklabels(xtick_labels)
ax[1].tick_params(axis='y', labelsize=14)
# Set 5 y-ticks and y-lim automatically based on data
ax[1].set_ylim(0, np.ceil(y_max_sh * 1.1 * 10) / 10 if y_max_sh > 0 else 1)
ax[1].yaxis.set_major_locator(plt.MaxNLocator(nbins=5))

patches = [mpatches.Patch(color=color_map[protocol], label=protocol) for protocol in protocols]
fig.legend(
    handles=patches,
    loc='upper center',
    ncol=4,
    bbox_to_anchor=(0.5, 1.28),
    fontsize=16,
    columnspacing=1  # Increased spacing between labels
)
fig.savefig('packet_index.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Reproduction of Fig. 6 BSM visualization

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from matplotlib.ticker import FuncFormatter

def human_format(num, pos=None):
    # Formats number precisely in K, M, G etc. (including "1.5K", "2.7M", etc.)
    if num is None:
        return ""
    num = float(num)
    if num >= 1_000_000_000:
        val = num / 1_000_000_000
        return f'{val:.1f}G' if not val.is_integer() else f'{int(val)}G'
    elif num >= 1_000_000:
        val = num / 1_000_000
        return f'{val:.1f}M' if not val.is_integer() else f'{int(val)}M'
    elif num >= 1:
        val = num / 1_000
        return f'{val:.1f}K' if not val.is_integer() else f'{int(val)}K'
    else:
        return str(int(num)) if num.is_integer() else f"{num:.1f}"

color_map = {"NoProxy": "#4C78A8", "VMess": "#B279A2", "Shadowsocks": "#54A24B", "Trojan": "#F58518"}
protocol_markers = {"NoProxy": "o", "VMess": "s", "Shadowsocks": "^", "Trojan": "D"}

names = [
    [
        "www.office.com_js.monitor.azure.com",  # Short
        "www.cnblogs.com_cdn-www.cnblogs.com", # Short
    ],
    [
        "www.fromgeek.com_www.fromgeek.com",  # Long
        "www.apache.org_www.apache.org",  # Long
    ]
]

index = [
    ['a', 'b'],
    ['c', 'd'],
]

output_root = pa3_path("VisualSeg")
base = "tcp"


fig, axs = plt.subplots(2, 2, figsize=(12, 7))
fig.subplots_adjust(wspace=0.4)

for i, row in enumerate(names):
    for j, name in enumerate(row):
        ax = axs[i][j]
        cv_means = {}
        all_y_values = []  # Collect all Y values for y-lim autoscale

        for protocol in color_map:
            avg_array_path = Path(f"{output_root}/seg_compute/{base}/{protocol.lower()}/avg_{name}.npy")
            std_array_path = Path(f"{output_root}/seg_compute/{base}/{protocol.lower()}/std_{name}.npy")
            avg_byte_segments, std_byte_segments = np.load(avg_array_path), np.load(std_array_path)

            # mid = len(avg_byte_segments) // 2
            mid = 200_000
            if i == 1:
                avg_byte_segments = avg_byte_segments[mid:]
                std_byte_segments = std_byte_segments[mid:]
                x = np.arange(mid, mid + len(avg_byte_segments))
            else:
                x = np.arange(len(avg_byte_segments))

            cv_means[protocol] = float(np.mean(std_byte_segments/avg_byte_segments))
            marker_step = max(1, len(avg_byte_segments) // 10)
            ax.plot(
                x, avg_byte_segments,
                linewidth=2,
                color=color_map[protocol],
                label=f"{protocol}",
                marker=protocol_markers[protocol],
                markevery=marker_step,
                markersize=6,
            )
            all_y_values.extend(avg_byte_segments)  # Collect plotted values for y-lim

        # Remove per-subplot labels; set global labels after the loop
        host, sni = name.split('_')
        # The first row represents host transmitting short flows, the second row represents those transmitting long flows
        ax.set_title(
            f'({index[i][j]}) Host: {sni}' + (f' (Short)' if i == 0 else f' (Long)'),
            fontsize=20,
        )
        # Change xticks font size to 10
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        # Add human-readable byte tick formatter to x-axis
        ax.xaxis.set_major_formatter(FuncFormatter(human_format))

        # Set y-ticks as instructed
        if i == 0:
            ax.set_yticks([10, 30, 50, 70, 90])
        else:
            ax.set_yticks([400, 600, 800, 1000, 1200])

        max_cv = max(cv_means.values())

        handles, labels = ax.get_legend_handles_labels()

fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02), fontsize=20)
# Set global labels after the loop
fig.supxlabel('Byte Index', y=0.02, fontsize=20)
fig.supylabel('Intra-flow Segment Index', x=0.04, fontsize=20)
fig.subplots_adjust(hspace=0.3)

fig.savefig('byte_segment_map.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Reproduction of Fig. 9 WF performance degradation.
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

# Set global font to Times New Roman (if available)
matplotlib.rcParams['font.family'] = 'serif'

# Define the matrices
DF = np.array([
    [0.99, 0.52, 0.48],
    [0.49, 0.99, 0.46],
    [0.49, 0.56, 0.98]
])

BAPM = np.array([
    [0.98, 0.38, 0.43],
    [0.36, 0.98, 0.45],
    [0.39, 0.36, 0.94]
])

TF = np.array([
    [0.98, 0.91, 0.87],
    [0.90, 0.99, 0.88],
    [0.90, 0.92, 0.99]
])

NetCLR = np.array([
    [0.97, 0.33, 0.39],
    [0.37, 0.98, 0.42],
    [0.33, 0.32, 0.95]
])

TikTok = np.array([
    [0.98, 0.87, 0.78],
    [0.69, 0.98, 0.70],
    [0.74, 0.84, 0.96]
])

RF = np.array([
    [0.99, 0.63, 0.60],
    [0.45, 0.98, 0.49],
    [0.59, 0.67, 0.97]
])

matrices = [DF, BAPM, TF, NetCLR, TikTok, RF]
titles = ['DF', 'BAPM', 'TF', 'NetCLR', 'TikTok', 'RF']

def plot_6_heatmaps(matrices, titles, fontsize=12, hspace=0.3, wspace=0.15, output_path="perf_degrade.pdf"):
    """
    Plot six 3x3 matrices in a 2x3 grid as heatmaps with shared colorbar.

    Args:
        matrices (list of np.ndarray): Six 3x3 matrices.
        titles (list of str): Titles for the subplots.
        fontsize: Fontsize for annotation.
        hspace: Height (vertical) space between plots.
        wspace: Width (horizontal) space between plots.
    """
    from matplotlib import cm

    fig, axes = plt.subplots(2, 3, figsize=(10, 6))
    cmap = cm.get_cmap('viridis')
    vmin, vmax = 0.3, 1.0  # clip colormap as requested

    from matplotlib.colors import LinearSegmentedColormap

    # endpoint_colors = ["#E45756", "#fde724", "#9dd93a", "#1e998a", "#37598c"]
    # endpoint_colors = ["#de9090", "#62bd6e", "#6288bd"]
    endpoint_colors = ['#fb9b83', '#f0e1a8', '#7dcfaf', '#a9c4fe']
    endpoint_colors.reverse()
    custom_cmap = LinearSegmentedColormap.from_list('custom_f1', endpoint_colors, N=256)

    im_handles = []

    for idx, (mat, title) in enumerate(zip(matrices, titles)):
        ax = axes[idx // 3, idx % 3]
        im = ax.imshow(mat, cmap=custom_cmap, vmin=vmin, vmax=vmax)
        im_handles.append(im)
        # Annotate the cells with bold, Times New Roman numbers
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(
                    j, i, f'{mat[i, j]:.2f}',
                    ha='center',
                    va='center',
                    color='w',
                    fontsize=fontsize,
                    fontweight='bold',
                    fontfamily='serif'
                )
        ax.set_title(title, fontsize=fontsize + 2, fontfamily='serif')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.subplots_adjust(hspace=hspace, wspace=wspace)
    cbar = fig.colorbar(im_handles[0], ax=axes, orientation='vertical', fraction=0.025, pad=0.02)
    cbar.ax.tick_params(labelsize=fontsize)
    cbar.set_label("F1 Score", fontsize=fontsize, fontfamily='serif')
    plt.savefig(output_path, dpi=300, bbox_inches="tight", format='pdf')
    plt.show()
    

plot_6_heatmaps(matrices, titles, fontsize=16, hspace=0.18, wspace=0.01)

In [ ]:
# Reproduction of Fig. 10 BFT parameter selection.

import numpy as np
from pa3.tools.extractor import sni_cover
import pandas as pd

def bound_gen(*ranges: tuple):
    lower_bounds = np.array([r[0] for r in ranges])
    upper_bounds = np.array([r[1] for r in ranges])
    return lower_bounds, upper_bounds

def proportion_in_range(arr, lower_bounds, upper_bounds):
    """
    Calculate the proportion of the array that is in the range of the lower and upper bounds.
    """
    assert len(lower_bounds) == len(upper_bounds)
    in_idx = set()
    for (lower, upper) in zip(lower_bounds, upper_bounds):
        assert lower <= upper
        in_idx.update(np.where(np.logical_and(arr > lower, arr < upper))[0])
    return len(in_idx) / len(arr)

def get_utility(filtered, remained, mu):
    return (1 + mu) * filtered * remained / (mu * filtered + remained)


INTRINSIC_SNIS = ['firefox-settings-attachments.cdn.mozilla.net', 'firefox.settings.services.mozilla.com', 'content-signature-2.cdn.mozilla.net']
STAT_ROOT = "../data_extract"
PROTOCOLS = ["vmess", "trojan", "shadowsocks"]


website_flow_remaining = {"vmess": [], "trojan": [], "shadowsocks": []}
website_flow_small_remaining = {"vmess": [], "trojan": [], "shadowsocks": []}
BIH_flow_filtered = {"vmess": [], "trojan": [], "shadowsocks": []}
BIH_flow_large_filtered = {"vmess": [], "trojan": [], "shadowsocks": []}
BIH_flow_middle_filtered = {"vmess": [], "trojan": [], "shadowsocks": []}
BIH_flow_small_filtered = {"vmess": [], "trojan": [], "shadowsocks": []}
utility = {"vmess": [], "trojan": [], "shadowsocks": []}

for protocol in PROTOCOLS:
    df = pd.read_csv(f"{STAT_ROOT}/firefox-settings-attachments.cdn.mozilla.net.csv")
    attachments_flow = df[protocol].dropna().to_numpy().astype(np.int64)
    df = pd.read_csv(f"{STAT_ROOT}/firefox.settings.services.mozilla.com.csv")
    services_flow = df[protocol].dropna().to_numpy().astype(np.int64)
    df = pd.read_csv(f"{STAT_ROOT}/content-signature-2.cdn.mozilla.net.csv")
    signature_flow = df[protocol].dropna().to_numpy().astype(np.int64)
    # Data preparation
    website_flow = pa3_path(f"VisualSeg/flow_size/{protocol}.npz")
    website_flow = np.load(website_flow)['flow_size']
    website_flow_small = website_flow[website_flow < 30_000]

    # Concatenate the flows of the three SNIs.
    BIH_flow = np.concatenate([attachments_flow, services_flow, signature_flow])
    BIH_flow_large = attachments_flow[(attachments_flow > 2_000_000) & (attachments_flow <= 3_000_000)]
    BIH_flow_middle = signature_flow[(signature_flow > 80_000) & (signature_flow <= 120_000)]
    BIH_flow_small = services_flow[services_flow <= 30_000]
    for coverage in range(11):
        # Given coverage, we first calculate the total cover of the intrinsic SNIs.
        total_cover = []
        for sni in INTRINSIC_SNIS:
            cover, _ = sni_cover(STAT_ROOT, protocol, sni, float(coverage)/10)
            total_cover += cover
        lower_bounds, upper_bounds = bound_gen(*total_cover)

        website_flow_remaining[protocol].append(1 - proportion_in_range(website_flow, lower_bounds, upper_bounds))
        website_flow_small_remaining[protocol].append(1 - proportion_in_range(website_flow_small, lower_bounds, upper_bounds))
        BIH_flow_filtered[protocol].append(proportion_in_range(BIH_flow, lower_bounds, upper_bounds))
        BIH_flow_large_filtered[protocol].append(proportion_in_range(BIH_flow_large, lower_bounds, upper_bounds))
        BIH_flow_middle_filtered[protocol].append(proportion_in_range(BIH_flow_middle, lower_bounds, upper_bounds))
        BIH_flow_small_filtered[protocol].append(proportion_in_range(BIH_flow_small, lower_bounds, upper_bounds))
        utility[protocol].append(get_utility(BIH_flow_large_filtered[protocol][-1], website_flow_small_remaining[protocol][-1], 5))


# After collecting the statistics, draw a 1x3 plot for the arrays.
import matplotlib.pyplot as plt

coverage_range = [i/10 for i in range(11)]
array_dicts = [
    website_flow_small_remaining,
    website_flow_remaining,
    BIH_flow_small_filtered,
    BIH_flow_filtered,
    BIH_flow_large_filtered,
]
utility_dict = utility

array_labels = [
    r"Website Flow Retained (<30KB)",
    "Website Flow Retained",
    r"BIH Flow Filtered (<30KB)",
    "BIH Flow Filtered",
    r"BIH Flow Filtered (>1M)",
]
marker_dict = ['o', 's', 'v', 'P', 'X']
color_dict = ['#4C78A8', '#72B7B2', '#54A24B', '#9D755D', '#B279A2']
utility_marker = 'D'

protocols_display = ['VMess', 'Shadowsocks', 'Trojan']

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3), sharey=True)
utility_axes = [None] * 3

lines = []
labels = []
utility_lines = []
utility_labels = []

for idx, protocol in enumerate(protocols_display):
    ax = axes[idx]
    protocol_key = protocol.lower()
    these_lines = []
    # Plot all arrays except Utility
    for arr_idx, array in enumerate(array_dicts):
        line, = ax.plot(
            coverage_range,
            array[protocol_key],
            marker=marker_dict[arr_idx],
            label=array_labels[arr_idx],
            color=color_dict[arr_idx]
        )
        # Only populate shared legend lines/labels once
        if idx == 0:
            if arr_idx < len(lines):
                continue
            lines.append(line)
            labels.append(array_labels[arr_idx])
        these_lines.append(line)

    # For every subplot, plot utility on the right y-axis
    utility_axes[idx] = ax.twinx()
    line_utility, = utility_axes[idx].plot(
        coverage_range,
        utility_dict[protocol_key],
        marker=utility_marker,
        linestyle="--",
        color='#E45756',
        label=r"$U(\alpha)$"
    )
    utility_lines.append(line_utility)
    if idx == 2:
        utility_labels.append(r"Balance score $U(\alpha)$")
    elif idx == 0:
        # Add one Utility label for the legend; only once
        utility_labels.append("Utility")
    else:
        utility_labels.append("")

    # --- Draw optimal coverage vertical line and annotation ---
    util_arr = utility_dict[protocol_key]
    util_arr_np = np.asarray(util_arr)
    max_util = np.max(util_arr_np)
    optimal_idx = np.where(util_arr_np == max_util)[0][0]  # the first max
    optimal_coverage = coverage_range[optimal_idx]

    # Draw the vertical line at optimal_coverage (across the utility axis)
    ax_vspan = utility_axes[idx].axvline(optimal_coverage, color='red', linestyle=':', linewidth=2, alpha=0.7)

    # Annotate the optimal coverage at the bottom left side of the line
    ymin, ymax = utility_axes[idx].get_ylim()
    # We want to place the text near the bottom, left side of the line
    y_annotate = ymin + 0.02 * (ymax - ymin)
    utility_axes[idx].annotate(
        rf'$\alpha^*={optimal_coverage:.1f}$',
        xy=(optimal_coverage, y_annotate),
        xytext=(-2, 4),
        textcoords='offset points',
        ha='right',
        va='bottom',
        color='black',
        fontsize=16,
    )

    ax.set_title(protocol, fontsize=18)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    ax.set_xlim(0, 1)
    # Set the x-axis ticks to [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax.set_xlim(-0.05, 1.05)
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'])
    if idx == 1:
        ax.set_xlabel(r"Coverage Threshold $\alpha$", fontsize=20)
    if idx == 0:
        ax.set_ylabel("Flow Proportion", fontsize=20)
    if idx == 2:
        utility_axes[idx].set_ylabel(r"Balance Score $U(\alpha)$", fontsize=18)
        utility_axes[idx].tick_params(axis='y', labelsize=18)
        utility_axes[idx].set_yticks([0, 0.2, 0.4, 0.6, 0.8])
        utility_axes[idx].set_yticklabels(['0', '0.2', '0.4', '0.6', '0.8'])
        # No y-label for utility axes except last one
    else:
        utility_axes[idx].set_yticklabels([])

# Build handles for the legend, using lines from left axis (first protocol), and utility from right axes (one label)
# Only use one Utility curve/label to avoid duplicate entries
all_lines = lines + [utility_lines[0]]
all_labels = labels + [r"$U(\alpha)$"]

# Place the shared legend below all the subplots
fig.legend(
    all_lines, 
    all_labels, 
    loc='upper center', 
    ncol=3, 
    bbox_to_anchor=(0.5, 1.26), 
    fontsize=16,
    columnspacing=.4
    )
plt.tight_layout()
fig.savefig("coverage.pdf", bbox_inches='tight', dpi=300, format='pdf')
plt.show()

In [ ]:
# Reproduction of Fig. 11 PHT parameter selection

import pickle

from pa3.tools.extractor import *
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def fit_uniform_binner(v):
    return PacketSizeBinner.fit_uniform(LOWER_BOUND, UPPER_BOUND, v)

UPPER_BOUND = 1500
LOWER_BOUND = -1500
WINDOW_SIZE = 2
PROTOCOLS = ['vmess', 'shadowsocks', 'trojan']
PROTOCOLS_DISPLAY = {'vmess': 'VMess', 'shadowsocks': 'Shadowsocks', 'trojan': 'Trojan'}

STRIP_INDICES = {'vmess': [3, 4], 'shadowsocks': [4, 5], 'trojan': [3, 4]}
LAMBDA_PENALTY = {'vmess': 0.001, 'shadowsocks': 0.01, 'trojan': 0.002}
TRAIN_SAMPLES = {'vmess': 400, 'shadowsocks': 40, 'trojan': 4}
VOCABULARY_RANGE = [1, 2, 4, 7, 9, 11, 30, 50, 70, 90, 101, 300, 500, 700, 900, 1001, 1400, 1800, 2000, 2400, 2800]

precision, recall = {'vmess': [], 'shadowsocks': [], 'trojan': []}, {'vmess': [], 'shadowsocks': [], 'trojan': []}
objective = dict()
vocab_star = dict()

for protocol in PROTOCOLS:
    # Data preparation
    with open(f'{protocol}.pkl', 'rb') as f:
        flows = pickle.load(f)

    for vocabulary_size in VOCABULARY_RANGE:
        # vocabulary_sizes.append(vocabulary_size)
        flows_train = flows[:TRAIN_SAMPLES[protocol]]
        flows_test = flows[4000:]
        window_type = set()
        for flow in flows_test:
            binned_flow = uniform_bin(flow, LOWER_BOUND, UPPER_BOUND, vocabulary_size)
            window_type.add(tuple(binned_flow[STRIP_INDICES[protocol]]))

        binner = PacketSizeBinner.fit_uniform(
            LOWER_BOUND, UPPER_BOUND, vocabulary_size=vocabulary_size
        )
        # binner = PacketSizeBinner.fit_distribution(packet_size_count, vocabulary_size)

        ngram_db = train_ngram_db(
            flows_train, STRIP_INDICES[protocol], WINDOW_SIZE, binner
        )
        p, r = evaluate_ngram(
            flows_test, STRIP_INDICES[protocol], WINDOW_SIZE, ngram_db, binner
        )
        precision[protocol].append(p)
        recall[protocol].append(r)

    rows = sweep_vocabulary_objective(
        flows[:4000],
        STRIP_INDICES[protocol],
        WINDOW_SIZE,
        VOCABULARY_RANGE,
        fit_binner=fit_uniform_binner,
        lambda_penalty=LAMBDA_PENALTY[protocol],
    )
    df_obj = pd.DataFrame(rows)

    objective[protocol] = df_obj["objective"]
    vocab_star[protocol] = int(df_obj.loc[df_obj["objective"].idxmax(), "vocabulary_size"])

fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharex=True)
color_f1 = "#4C78A8",  # blue
color_obj = "#E45756",  # red
f1_ticks = [0, .2, .4, .6, .8, 1.0]

# We'll store the twin axes to access them later if needed
secondary_axes = []

for i, (ax1, protocol) in enumerate(zip(axes, PROTOCOLS)):
    f1_scores = [
        2 * p * r / (p + r) if (p + r) > 0 else 0.0
        for p, r in zip(precision[protocol], recall[protocol])
    ]
    lns1 = ax1.plot(
        VOCABULARY_RANGE,
        f1_scores,
        marker="o",
        markersize=5,
        color="#4C78A8",  # blue
        linestyle="-",
        label="F1 Score",
    )
    ax1.set_xscale("log")
    ax1.axvline(
        vocab_star[protocol],
        color="#4C78A8",  # blue
        linestyle="--",
        # label=f"|V|*={vocab_star[protocol]}",
    )

    # Only set ylabel and show y-ticks on the left axis of the first subplot
    if i == 0:
        ax1.set_ylabel("F1-Score", color="#4C78A8", fontsize=18)
        ax1.tick_params(axis="y", labelcolor="#4C78A8", labelsize=16, left=True, labelleft=True)
        ax1.set_yticks(f1_ticks)
        ax1.set_yticklabels([str(f1) for f1 in f1_ticks])
    else:
        ax1.set_ylabel("")
        ax1.tick_params(axis="y", labelleft=False, left=False)
    ax1.tick_params(axis="x", labelsize=12)

    ax2 = ax1.twinx()
    secondary_axes.append(ax2)
    lns2 = ax2.plot(
        VOCABULARY_RANGE,
        objective[protocol],
        marker="s",
        markersize=5,
        color="#E45756",
        linestyle="--",
        label="MI Objective",
    )

    ax1.tick_params(axis="x", labelsize=16)
    ax1.set_xticks([1e0, 1e1, 1e2, 1e3])
    ax1.set_xticklabels([r"$10^0$", r"$10^1$", r"$10^2$", r"$10^3$"])

    # Only set ylabel and show y-ticks on the right axis of the last subplot
    if i == len(axes) - 1:
        ax2.set_ylabel("Objective Value", color="#E45756", fontsize=18)
        ax2.tick_params(axis="y", labelcolor="#E45756", labelsize=16, right=True, labelright=True)
    else:
        ax2.set_ylabel("")
        ax2.tick_params(axis="y", labelright=False, right=False)

    if i == 1:
        ax1.set_xlabel(r"Vocabulary size $v$", fontsize=18)

    ax1.set_title(PROTOCOLS_DISPLAY[protocol], fontsize=18)
    ax1.annotate(
        rf"$v^*={vocab_star[protocol]}$",
        xy=(vocab_star[protocol], 0.04),
        xycoords=("data", "axes fraction"),
        xytext=(2, 0),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=18,
        bbox=dict(boxstyle="round,pad=0.2", fc="none", ec="none", alpha=0.7),
    )

legend_elems = [
    Line2D([0], [0], color="#4C78A8", marker="o", linestyle="-", label="Detection Performace"),
    Line2D([0], [0], color="#E45756", marker="s", linestyle="--", label=r"MI Objective $J(v)$"),
    # Line2D([0], [0], color="gray", linestyle="--", label="|V|*"),
]
fig.legend(
    handles=legend_elems,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.08),
    fontsize=18,
)

plt.tight_layout(rect=[0, 0.02, 1, 0.92])
fig.savefig("optimal_vocab_size.pdf", bbox_inches='tight', dpi=300, format='pdf')
plt.show()

In [ ]:
# Reproduction of Fig. 12 BFT ranges for different browser versions.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pa3.utils.statistics import greedy_mass_covering

versions = [v for v in range(132, 142)]
protocols = ["VMess", "Shadowsocks", "Trojan"]
color_map = ["#B279A2", "#54A24B", "#F58518"]
coverage = .4

SNI_BIN_SIZE = {
    'firefox-settings-attachments.cdn.mozilla.net': 20000, 
    'firefox.settings.services.mozilla.com': 500, 
    'content-signature-2.cdn.mozilla.net': 1000
}

fig, axs = plt.subplots(1, 3, figsize=(8, 2), constrained_layout=True, sharey=True)
patches = [mpatches.Patch(color=color_map[i], label=protocols[i]) for i in range(3)]
y_ticks = [0, .5e7, 1e7, 1.5e7]
axs[0].set_xticks(versions)
axs[0].set_ylabel("Size", fontsize=16)
axs[0].set_yticks(y_ticks)
axs[0].tick_params(axis='x', rotation=45, labelsize=14)
axs[0].tick_params(axis='y', labelsize=14)
axs[0].set_title("VMess", fontsize=16)
axs[1].set_xticks(versions)
axs[1].set_yticks(y_ticks)
axs[1].tick_params(axis='x', rotation=45, labelsize=14)
axs[1].set_title("Shadowsocks", fontsize=16)
axs[1].set_xlabel("Firefox Version", fontsize=16)
axs[1].tick_params(axis='y', labelleft=False)
axs[2].set_xticks(versions)
axs[2].set_yticks(y_ticks)
axs[2].tick_params(axis='x', rotation=45, labelsize=14)
axs[2].set_title("Trojan", fontsize=16)
axs[2].tick_params(axis='y', labelleft=False)

for version in versions:
    dfs = dict()
    dfs['content-signature-2.cdn.mozilla.net'] = pd.read_csv(pa3_path(f"VisualSeg/firefox_dataset/{version}/content-signature-2.cdn.mozilla.net.csv"))
    dfs['firefox.settings.services.mozilla.com'] = pd.read_csv(pa3_path(f"VisualSeg/firefox_dataset/{version}/firefox.settings.services.mozilla.com.csv"))
    dfs['firefox-settings-attachments.cdn.mozilla.net'] = pd.read_csv(pa3_path(f"VisualSeg/firefox_dataset/{version}/firefox-settings-attachments.cdn.mozilla.net.csv"))
    for i, protocol in enumerate(protocols):
        total_cover = []
        for sni in dfs:
            array = dfs[sni][protocol.lower()].dropna().to_numpy().astype(np.int64)
            cover, actual_coverage = greedy_mass_covering(array, SNI_BIN_SIZE[sni], coverage)
            total_cover += cover
        
        for cover in total_cover:
            axs[i].scatter(version, (cover[0] + cover[1])/2, color=color_map[i], marker='o')

# fig.legend(handles=patches, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.21), fontsize=14)
plt.savefig('version.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Reproduction of Fig. 13 Stability of Gaussian parameter estimation

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def repeated_estimation_errors(x, k_list, n_repeat=10, rng=None, dispersion="std", ddof=0):
    """
    Use the full data as reference, then repeatedly sample k points without replacement.
    
    Returns:
        mean_errors: list of arrays, one array for each k
        disp_errors: list of arrays, one array for each k
        ref: dictionary containing full-data statistics
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.asarray(x).reshape(-1)
    n = len(x)

    mu_all = np.mean(x)
    std_all = np.std(x, ddof=ddof)
    var_all = np.var(x, ddof=ddof)

    mean_errors = []
    disp_errors = []

    for k in k_list:
        if k > n:
            raise ValueError(f"k={k} is larger than data size n={n}.")

        cur_mean_errors = []
        cur_disp_errors = []

        for _ in range(n_repeat):
            idx = rng.choice(n, size=k, replace=False)
            sampled = x[idx]

            mu_k = np.mean(sampled)
            std_k = np.std(sampled, ddof=ddof)
            var_k = np.var(sampled, ddof=ddof)

            # Mean error normalized by full-data std.
            e_mu = abs(mu_k - mu_all) / mu_all

            if dispersion == "std":
                e_disp = abs(std_k - std_all) / std_all
            elif dispersion == "var":
                if var_all == 0:
                    raise ValueError("Full-data variance is zero; variance error is undefined.")
                e_disp = abs(var_k - var_all) / var_all
            else:
                raise ValueError("dispersion must be either 'std' or 'var'.")

            cur_mean_errors.append(e_mu)
            cur_disp_errors.append(e_disp)

        mean_errors.append(np.asarray(cur_mean_errors))
        disp_errors.append(np.asarray(cur_disp_errors))

    ref = {
        "n": n,
        "mu_all": mu_all,
        "std_all": std_all,
        "var_all": var_all,
    }

    return mean_errors, disp_errors, ref

SEED = 114514
N_REPEAT = 200
K_LIST = [30, 27, 24, 21, 18, 15]
DISPERSION = "std"
DDOF = 0

rng = np.random.default_rng(SEED)

shadowsocks_vmess_ratio = np.load(
    pa3_path("VisualSeg/slope/shadowsocks_vmess_slope_ratio.npz")
)["slope_ratio"]

trojan_vmess_ratio = np.load(
    pa3_path("VisualSeg/slope/trojan_vmess_slope_ratio.npz")
)["slope_ratio"]

shadowsocks_vmess_ratio = np.asarray(shadowsocks_vmess_ratio).reshape(-1)
trojan_vmess_ratio = np.asarray(trojan_vmess_ratio).reshape(-1)

ss_mean_err, ss_disp_err, ss_ref = repeated_estimation_errors(
    shadowsocks_vmess_ratio,
    K_LIST,
    n_repeat=N_REPEAT,
    rng=rng,
    dispersion=DISPERSION,
    ddof=DDOF,
)

ss_mu, ss_std = ss_ref["mu_all"], ss_ref["std_all"]

tr_mean_err, tr_disp_err, tr_ref = repeated_estimation_errors(
    trojan_vmess_ratio,
    K_LIST,
    n_repeat=N_REPEAT,
    rng=rng,
    dispersion=DISPERSION,
    ddof=DDOF,
)

tr_mu, tr_std = tr_ref["mu_all"], tr_ref["std_all"]

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6), sharey=True)

def plot_grouped_boxplot(ax, mean_errors, disp_errors, k_list, title, dispersion="std", ylabel=None, show_legend=False, set_xlabel=False):
    base_pos = np.arange(len(k_list))
    offset = 0.18
    width = 0.30

    mean_pos = base_pos - offset
    disp_pos = base_pos + offset

    bp_mean = ax.boxplot(
        mean_errors,
        positions=mean_pos,
        widths=width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(linewidth=1.2),
        boxprops=dict(linewidth=1.0),
        whiskerprops=dict(linewidth=1.0),
        capprops=dict(linewidth=1.0),
    )

    bp_disp = ax.boxplot(
        disp_errors,
        positions=disp_pos,
        widths=width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(linewidth=1.2),
        boxprops=dict(linewidth=1.0),
        whiskerprops=dict(linewidth=1.0),
        capprops=dict(linewidth=1.0),
    )

    for box in bp_mean["boxes"]:
        box.set_facecolor("lightgray")
        box.set_hatch("//")

    for box in bp_disp["boxes"]:
        box.set_facecolor("#fbf3d5")
        box.set_hatch("\\\\")

    ax.axhline(0, linestyle="--", linewidth=0.8)

    ax.set_title(title, fontsize=14)
    ax.set_xticks(base_pos)
    ax.set_xticklabels([10, 9, 8, 7, 6, 5], fontsize=14)
    # Do not set xlabel in this function, instead do it after subplots
    if set_xlabel:
        ax.set_xlabel("Number of Websites", fontsize=14)
    ax.set_ylim(top=1)
    ax.set_yticks([0, .2, .4, .6, .8, 1.0])
    ax.set_yticklabels([0, .2, .4, .6, .8, 1.0], fontsize=12)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=14)

    disp_label = r"$e_{\sigma}(k)$" if dispersion == "std" else r"$e_{\sigma^2}$"

    legend_handles = [
        Patch(facecolor="lightgray", hatch="//", label=r"$e_{\mu}(k)$"),
        Patch(facecolor="#fbf3d5", hatch="\\\\", label=disp_label),
    ]
    if show_legend:
        # Only create the legend for the provided axis
        ax.legend(handles=legend_handles, frameon=False, fontsize=14)
    return legend_handles

# Disable x-label in both subplots, add it as shared x-label to fig after
legend_handles = plot_grouped_boxplot(
    axes[0],
    ss_mean_err,
    ss_disp_err,
    K_LIST,
    "Shadowsocks",
    dispersion=DISPERSION,
    ylabel="Normalized Error",
    show_legend=False,
    set_xlabel=False
)

plot_grouped_boxplot(
    axes[1],
    tr_mean_err,
    tr_disp_err,
    K_LIST,
    "Trojan",
    dispersion=DISPERSION,
    ylabel=None,
    show_legend=False,
    set_xlabel=False
)

# Set a shared x-label for both subplots
fig.supxlabel(r"Number of Sampled Websites $(k)$", fontsize=14, x=0.5, y=0.1)

# Annotate Shadowsocks mean/std to upper left of first subplot
axes[0].text(
    0.02, 0.96, 
    r"$\mu_u$ = {:.3f}".format(ss_mu) + "\n" + r"$\sigma_u$ = {:.3f}".format(ss_std),
    transform=axes[0].transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='left',
    bbox=dict(facecolor='white', alpha=0.7, pad=2.0)
)

# Annotate Trojan mean/std to upper left of second subplot
axes[1].text(
    0.02, 0.96, 
    r"$\mu_u$ = {:.3f}".format(tr_mu) + "\n" + r"$\sigma_u$ = {:.3f}".format(tr_std),
    transform=axes[1].transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='left',
    bbox=dict(facecolor='white', alpha=0.7, pad=2.0)
)

# Place the shared legend above the subplots
disp_label = r"Std. Error $e_{\sigma}(k)$" if DISPERSION == "std" else r"$e_{\sigma^2}$"
shared_legend_handles = [
    Patch(facecolor="lightgray", hatch="//", label=r"Mean Error $e_{\mu}(k)$"),
    Patch(facecolor="#fbf3d5", hatch="\\\\", label=disp_label),
]
fig.legend(
    handles=shared_legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.55, 1.13),
    ncol=2,
    frameon=False,
    fontsize=14,
)

plt.tight_layout()
plt.savefig("gaussian_param_estimation.pdf", bbox_inches="tight")
plt.show()